# S7 · Qwen original, few-shot y LoRA
Datos sintéticos. El mismo modelo y protocolo en las tres condiciones. La inferencia y el entrenamiento se activan explícitamente; los datos y la configuración pueden inspeccionarse sin GPU.

**Cómo leerlo:** ejecuta primero las celdas de datos e inspección. `RUN_LOCAL_INFERENCE` activa carga/consultas; `RUN_TRAINING` activa el proceso que modifica y guarda un adaptador. Mantén ambos en `False` para explicar el código sin ejecutarlo. Las funciones compartidas viven en `curso/` y la receta real en `scripts/entrenar_lora.py`.

In [ ]:
# Path permite construir rutas, comprobar archivos y leer/escribir sin concatenar separadores a mano.
from pathlib import Path

# Gestiona rutas de importación y el intérprete actual de Python.
# Convierte entre texto JSON y diccionarios/listas Python, manteniendo respuestas como evidencia.
import sys, json

# Raíz del proyecto utilizada para resolver las rutas de datos y scripts.
ROOT = Path.cwd()
# Raíz del proyecto utilizada para resolver las rutas de datos y scripts.
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
# Añade una ruta de importación para poder reutilizar el paquete curso y los scripts del proyecto.
sys.path.insert(0, str(ROOT))
# Reutiliza funciones del núcleo compartido para mantener el mismo protocolo entre scripts.
from curso.common import (
    read_jsonl,
    validate_splits,
    assert_frozen,
    MODEL_ID,
    REVISION,
    messages,
)

# Carpeta del adaptador que se entrenará o recargará; puede apuntar a un ensayo nuevo.
ADAPTER = ROOT / "modelos/lora-soporte"
# Interruptor explícito de consultas locales; False permite leer el notebook sin cargar pesos.
RUN_LOCAL_INFERENCE = False
# Interruptor explícito del entrenamiento; False evita iniciarlo al ejecutar todas las celdas.
RUN_TRAINING = False
validate_splits(), assert_frozen()

## 1. Datos
240 train, 30 validación y 30 test, separados por familia. Test no se utiliza para mejorar el sistema. Los prefijos dan diversidad limitada.

Cada fila tiene `incidencia` y `objetivo`. El objetivo contiene el JSON que queremos enseñar: categoría, prioridad y una respuesta breve. `id` y `familia` sirven para trazabilidad y separación, y no se incluyen como texto del modelo. El archivo JSONL guarda una fila por línea; `read_jsonl` lo convierte en una lista de diccionarios.

In [ ]:
# Ejemplos leídos del archivo de entrenamiento para inspección o preparación del notebook.
train = read_jsonl(ROOT / "datos/train.jsonl")
# Muestra la primera fila real de train sin ejecutar entrenamiento.
train[0]

## 2. Chat template
Con el modelo descargado, activar RUN_LOCAL_INFERENCE permite ver su template. Thinking off y tokenización sin añadir de nuevo tokens especiales.

`messages` construye system + user. `render` aplica las marcas de roles de Qwen y prepara el turno assistant. El tokenizer convierte ese texto en IDs; `add_special_tokens=False` evita añadir otra vez las marcas que ya puso el template. En entrenamiento se añade también la respuesta objetivo.

In [ ]:
# Solo carga/consulta el modelo cuando se ha activado este interruptor explícitamente.
if RUN_LOCAL_INFERENCE:
    # Carga modelos/tokenizers y aporta generación y utilidades de entrenamiento.
    from transformers import AutoTokenizer

    # Reutiliza funciones del núcleo compartido para mantener el mismo protocolo entre scripts.
    from curso.qwen import render

    # Tokenizer del modelo: convierte el texto templado en IDs y decodifica tokens de salida.
    tok = AutoTokenizer.from_pretrained(
        MODEL_ID, revision=REVISION, token=False, trust_remote_code=False
    )
    # Texto formateado o decodificado que se pasa al siguiente paso del recorrido.
    text = render(tok, messages(train[0]["incidencia"]))
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print(text)
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print("Tokens:", len(tok(text, add_special_tokens=False)["input_ids"]))
else:
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print(
        "Template pendiente de activar inferencia local; descargar_modelo.py prepara el checkpoint."
    )

## 3. Few-shot
Cuatro ejemplos de train fijados antes de test. No se cambian los pesos.

Few-shot coloca cuatro parejas user/assistant en el contexto de la consulta nueva. Los IDs se leen de `prompts/fewshot-ids.json`, y los registros vienen exclusivamente de train. Esos ejemplos consumen tokens de entrada pero no ejecutan un optimizador ni generan un adaptador.

In [ ]:
# Reutiliza funciones del núcleo compartido para mantener el mismo protocolo entre scripts.
from curso.common import fewshot_ids

fewshot_ids(), messages("Tengo un cobro repetido en mi matrícula.", fewshot=True)

## 4. Original
Con RUN_LOCAL_INFERENCE=True se consulta el checkpoint original, sin adaptador.

`load_qwen()` carga la base sin adaptador. `generate` utiliza el mismo prompt y generación greedy que la condición LoRA. Al terminar liberamos referencias y caché del dispositivo antes de cargar otra condición. El texto se conserva aunque se equivoque.

In [ ]:
# Solo carga/consulta el modelo cuando se ha activado este interruptor explícitamente.
if RUN_LOCAL_INFERENCE:
    # Reutiliza funciones del núcleo compartido para mantener el mismo protocolo entre scripts.
    from curso.qwen import load_qwen, generate

    base_model, tok, device = load_qwen()
    # Respuesta del modelo base sin aplicar un adaptador.
    original = generate(
        base_model, tok, device, "Tengo un cobro repetido en mi matrícula."
    )
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print(original["respuesta_original"])
    del base_model, tok
    # Recoge objetos liberados antes de cargar otra condición del modelo en el notebook.
    # Motor de tensores y gradientes; durante inferencia se desactivan los gradientes.
    import gc, torch

    gc.collect()
    if device == "mps":
        torch.mps.empty_cache()
    elif device == "cuda":
        torch.cuda.empty_cache()
else:
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print("Original: inferencia no activada.")

## 5. Entrenamiento LoRA
La receta es fija: r=8, alpha=16, dropout=0.05, dos épocas y pérdida de completion. Cambia RUN_TRAINING a True para ejecutar. Una salida no vacía se rechaza; cambia ADAPTER para conservar ensayos anteriores.

El subprocess usa `sys.executable`, el mismo entorno de este notebook. La receta lee train/validación, crea mensajes y tokens, enmascara el prompt con `labels=-100` y entrega el Dataset a SFTTrainer. Este calcula pérdida del siguiente token, gradientes y actualizaciones LoRA. Dos épocas, batch 2 y acumulación 4 producen 60 pasos en el conjunto de 240 filas con un dispositivo. Si el destino ya contiene un adaptador, el comando se detiene para conservarlo.

In [ ]:
# Solo inicia entrenamiento si el profesor/alumno ha cambiado el interruptor a True.
if RUN_TRAINING:
    # Ejecuta el script con el mismo intérprete Python, respetando el entorno del notebook.
    import subprocess

    subprocess.run(
        [
            sys.executable,
            str(ROOT / "scripts/entrenar_lora.py"),
            "--salida",
            str(ADAPTER),
        ],
        cwd=ROOT,
        check=True,
    )
else:
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print(
        "Entrenamiento no iniciado. Comando: python scripts/entrenar_lora.py --salida modelos/lora-soporte"
    )

## 6. Recarga
Solo después de tener el adaptador. Se mantiene el mismo checkpoint, prompt y generación.

`load_qwen(str(ADAPTER))` carga primero la base y después PEFT aplica el adaptador. Sus pesos se combinan durante los cálculos de inferencia; no necesitamos copiar ni fusionar todo el modelo original. Consultar no entrena. `adapter_model.safetensors` contiene matrices aprendidas y `adapter_config.json` describe cómo cargarlas.

In [ ]:
# Comprueba presencia de archivo/carpeta antes de leerla o escribir; conserva la evidencia anterior.
if RUN_LOCAL_INFERENCE and (ADAPTER / "adapter_model.safetensors").exists():
    # Reutiliza funciones del núcleo compartido para mantener el mismo protocolo entre scripts.
    from curso.qwen import load_qwen, generate

    model, tok, device = load_qwen(str(ADAPTER))
    # Respuesta de la misma base con el adaptador solicitado activo.
    fitted = generate(model, tok, device, "Tengo un cobro repetido en mi matrícula.")
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print(fitted["respuesta_original"])
else:
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print("Recarga pendiente: activa inferencia y entrena un adaptador primero.")

## 7. Evaluación y referencias
Las cifras de referencia están fechadas y son anteriores. Los resultados nuevos se generan mediante evaluar_qwen.py y resumir.py --destino resultados/ensayo/informe. No retocar a partir de test.

Pérdida por token durante entrenamiento y acierto de categoría/prioridad son medidas distintas. `evaluar_qwen.py` genera respuestas libres con test y guarda originales; `resumir.py` calcula conteos y conserva errores en el denominador. El informe histórico identifica su fecha y entorno; una celda que lo muestra no vuelve a ejecutar aquel experimento.

In [ ]:
# Resumen por condición; distingue resultados medidos y condiciones pendientes.
report = ROOT / "resultados/ensayo/informe/resumen.json"
# Comprueba presencia de archivo/carpeta antes de leerla o escribir; conserva la evidencia anterior.
if report.exists():
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print("Informe local de esta ejecución:")
    # Resultados por condición, procedentes de un informe local o de la referencia anterior identificada.
    summary = json.loads(report.read_text())
else:
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print("Referencia anterior, NO una ejecución de este notebook:")
    # Evidencia histórica identificada por fecha y limitaciones; no es una ejecución nueva del notebook.
    reference = json.loads((ROOT / "docs/resultados-referencia.json").read_text())
    # Muestra el resultado/estado de este paso en la terminal; las claves no se añaden al informe.
    print(reference["fecha"], reference["limitaciones"])
    # Resultados por condición, procedentes de un informe local o de la referencia anterior identificada.
    summary = reference["resultados"]
{k: summary[k] for k in ("qwen_base", "qwen_fewshot", "qwen_lora")}

## 8. isbetter.ai
Con el adaptador listo: `python scripts/servir_comparacion.py --adaptador modelos/lora-soporte`. Local: `http://localhost:1235/v1`. Selecciona qwen-original y qwen-lora-soporte. Instrucciones: prompts/basico.txt; incidencia: Tengo un cobro repetido en mi matrícula. Las peticiones se serializan; evalúa respuestas, no un ranking de velocidad. Ver README y SECURITY.md.

El servidor carga una sola base con adaptador y expone dos alias. Para original desactiva temporalmente LoRA; para ajustado lo mantiene activo. Un lock protege ambas condiciones y las peticiones se ejecutan en serie. `/health` muestra revisión y hash del adaptador realmente cargado. El tiempo del navegador puede incluir espera en cola.